# Sentiment Classification des Leaders Politiques — Archelec
Classifie le sentiment exprimé dans les professions de foi envers le Président de la République en exercice.

**Pipeline :**
1. Annotation automatique (Claude Haiku) des 6 142 lignes restantes → `leaders_mentions_annotated.xlsx`
2. Fine-tuning CamemBERT (`camembert-base` → `CamembertForSequenceClassification`, 3 classes)
3. Évaluation : classification report + matrice de confusion
4. Analyse : sentiment par président, par année, par bloc politique → `leaders_sentiment_final.xlsx` + graphiques

## 0 — Installation des dépendances

In [ ]:
!pip install -q anthropic transformers torch accelerate scikit-learn openpyxl python-dotenv tqdm

## 1 — Imports & configuration

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from dotenv import load_dotenv
import anthropic

# ── Chemins ───────────────────────────────────────────────────────────────────
NOTEBOOK_DIR   = Path(os.getcwd())
ROOT_DIR       = NOTEBOOK_DIR.parent

OUTPUT_DIR     = ROOT_DIR / "data" / "results" / "output_best_model"
INPUT_FILE     = OUTPUT_DIR / "leaders_mentions.xlsx"
ANNOTATED_FILE = OUTPUT_DIR / "leaders_mentions_annotated.xlsx"
FINAL_FILE     = OUTPUT_DIR / "leaders_sentiment_final.xlsx"
MODEL_SAVE_DIR = ROOT_DIR / "models" / "sentiment_model_best"
GRAPHS_DIR     = OUTPUT_DIR / "graphs"

for d in [OUTPUT_DIR, MODEL_SAVE_DIR, GRAPHS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Clé API Anthropic ─────────────────────────────────────────────────────────
load_dotenv(ROOT_DIR / ".env")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY") or os.getenv("ANTHROPIC_KEY")
if not ANTHROPIC_KEY:
    raise ValueError(
        "Clé API introuvable. Définissez ANTHROPIC_API_KEY dans l'environnement "
        "ou dans le fichier .env à la racine du projet."
    )

LABEL_MAP  = {"négatif": 0, "neutre": 1, "positif": 2}
ID2LABEL   = {v: k for k, v in LABEL_MAP.items()}
LABEL_LIST = ["négatif", "neutre", "positif"]
COLORS     = {"négatif": "#e74c3c", "neutre": "#95a5a6", "positif": "#2ecc71"}

print("Chemins configurés :")
print(f"  INPUT      : {INPUT_FILE}")
print(f"  ANNOTATED  : {ANNOTATED_FILE}")
print(f"  FINAL      : {FINAL_FILE}")
print(f"  MODEL      : {MODEL_SAVE_DIR}")
print(f"  GRAPHS     : {GRAPHS_DIR}")


## 2 — Chargement des données

In [ ]:
df_input = pd.read_excel(INPUT_FILE)

print(f"Dimensions : {df_input.shape}")
print(f"Colonnes   : {df_input.columns.tolist()}")
print()
print("Répartition par président :")
print(df_input["actual_president"].value_counts().to_string())
print()
print("Sentiments déjà annotés :")
print(df_input["sentiment_president"].value_counts(dropna=False).to_string())
print()
already_done = df_input["sentiment_president"].isin(["positif", "négatif", "neutre"]).sum()
print(f"\nLignes déjà annotées : {already_done:,}")
print(f"Lignes restantes     : {len(df_input) - already_done:,}")
df_input.head(3)


## 3 — Annotation avec Claude API (Haiku) — 6 142 lignes restantes
Classe chaque extrait en **positif**, **négatif** ou **neutre** selon le sentiment exprimé envers le président.
Checkpoint automatique toutes les 100 lignes dans `leaders_mentions_annotated.xlsx`.

In [ ]:
def classify_sentiment(client: anthropic.Anthropic, context: str, president: str) -> str:
    """Classifie le sentiment envers le président : positif / négatif / neutre."""
    prompt = (
        f"Voici un extrait d'une profession de foi électorale française.\n"
        f"Le nom entre crochets désigne le Président de la République en exercice : {president}.\n\n"
        f"Extrait :\n{context}\n\n"
        f"Le candidat exprime-t-il un sentiment POSITIF (soutien, éloge, approbation), "
        f"NÉGATIF (critique, opposition, attaque) ou NEUTRE (simple mention factuelle, sans jugement) "
        f"envers {president} ?\n"
        f"Réponds uniquement avec un seul mot : positif, négatif ou neutre."
    )
    for attempt in range(3):
        try:
            msg = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=10,
                messages=[{"role": "user", "content": prompt}],
            )
            answer = msg.content[0].text.strip().lower()
            if "positif" in answer:
                return "positif"
            if "négatif" in answer or "negatif" in answer:
                return "négatif"
            if "neutre" in answer:
                return "neutre"
            return "neutre"
        except anthropic.RateLimitError:
            time.sleep(60)
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                print(f"[WARN] Échec après 3 tentatives : {e}")
                return "neutre"
    return "neutre"


In [ ]:
# Charger depuis checkpoint si disponible, sinon depuis l'original
if ANNOTATED_FILE.exists():
    df_work = pd.read_excel(ANNOTATED_FILE)
    print(f"Reprise depuis checkpoint : {ANNOTATED_FILE}")
else:
    df_work = df_input.copy()
    print("Démarrage depuis le fichier original")

# S'assurer que la colonne sentiment existe
if "sentiment_president" not in df_work.columns:
    df_work["sentiment_president"] = ""

# Identifier les lignes à annoter (vides ou NaN, et pas déjà 3-classes)
mask_todo = ~df_work["sentiment_president"].isin(["positif", "négatif", "neutre"])
to_annotate = df_work[mask_todo].index.tolist()

print(f"Déjà annotées : {(~mask_todo).sum():,}")
print(f"À annoter     : {len(to_annotate):,}")

CHECKPOINT_EVERY = 100
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

for i, idx in enumerate(tqdm(to_annotate, desc="Annotation Claude Haiku")):
    row = df_work.loc[idx]
    label = classify_sentiment(client, str(row["text"]), str(row["actual_president"]))
    df_work.at[idx, "sentiment_president"] = label

    if (i + 1) % CHECKPOINT_EVERY == 0:
        df_work.to_excel(ANNOTATED_FILE, index=False, engine="openpyxl")
        dist = df_work["sentiment_president"].value_counts().to_dict()
        print(f"  checkpoint {i+1}/{len(to_annotate)} — {dist}")

# Sauvegarde finale
df_work.to_excel(ANNOTATED_FILE, index=False, engine="openpyxl")
print(f"\nAnnotation terminée → {ANNOTATED_FILE}")
print("\nDistribution finale des sentiments :")
print(df_work["sentiment_president"].value_counts().to_string())


## 4 — Fine-tuning CamemBERT
Fine-tune `camembert-base` en classification de séquence (3 classes).
- Input : colonne `text` (~300 caractères de contexte, entité entre `[…]`)
- `max_length=256`, split stratifié 80/10/10
- Meilleur modèle sauvegardé dans `models/sentiment_model_best/`

In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ── Préparation des données ────────────────────────────────────────────────────
df_annotated = pd.read_excel(ANNOTATED_FILE)
df_clean = df_annotated[
    df_annotated["sentiment_president"].isin(["positif", "négatif", "neutre"])
].copy()
df_clean["label"] = df_clean["sentiment_president"].map(LABEL_MAP)

print(f"Dataset annoté : {len(df_clean):,} lignes")
print("Distribution des labels :")
print(df_clean["sentiment_president"].value_counts().to_string())

# Split stratifié 80/10/10
X_train, X_temp, y_train, y_temp = train_test_split(
    df_clean["text"].tolist(),
    df_clean["label"].tolist(),
    test_size=0.20,
    stratify=df_clean["label"].tolist(),
    random_state=42,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"\nSplit stratifié :")
print(f"  Train : {len(X_train):,}")
print(f"  Val   : {len(X_val):,}")
print(f"  Test  : {len(X_test):,}")


In [ ]:
MAX_LENGTH = 256
tokenizer  = CamembertTokenizer.from_pretrained("camembert-base")

class SentimentDataset(TorchDataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = SentimentDataset(X_train, y_train)
val_dataset   = SentimentDataset(X_val,   y_val)
test_dataset  = SentimentDataset(X_test,  y_test)
print("Datasets créés.")


In [ ]:
model_ft = CamembertForSequenceClassification.from_pretrained(
    "camembert-base",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL_MAP,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
    }

use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_dir=str(MODEL_SAVE_DIR / "logs"),
    logging_steps=50,
    report_to="none",
    fp16=use_fp16,
    warmup_ratio=0.1,
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Device : {'GPU (fp16)' if use_fp16 else 'CPU'}")
print(f"Paramètres du modèle : {sum(p.numel() for p in model_ft.parameters()):,}")
trainer.train()


In [ ]:
# Sauvegarde du meilleur modèle et du tokenizer
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))
print(f"Modèle sauvegardé : {MODEL_SAVE_DIR}")


## 5 — Évaluation
Classification report et matrice de confusion sur le **test set**.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Prédictions sur le test set
pred_output = trainer.predict(test_dataset)
y_pred  = np.argmax(pred_output.predictions, axis=-1)
y_true  = np.array(y_test)

y_pred_labels = [ID2LABEL[p] for p in y_pred]
y_true_labels = [ID2LABEL[t] for t in y_true]

print("=" * 55)
print("Classification Report — Test Set")
print("=" * 55)
print(classification_report(y_true_labels, y_pred_labels, target_names=LABEL_LIST))


In [ ]:
# ── Matrice de confusion ──────────────────────────────────────────────────────
cm = confusion_matrix(y_true_labels, y_pred_labels, labels=LABEL_LIST)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=LABEL_LIST, yticklabels=LABEL_LIST,
    ax=ax,
)
ax.set_xlabel("Prédiction")
ax.set_ylabel("Vérité terrain")
ax.set_title("Matrice de confusion — Sentiment (test set)")
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "confusion_matrix.png", dpi=150)
plt.show()
print(f"Graphique sauvegardé : {GRAPHS_DIR / 'confusion_matrix.png'}")


## 6 — Analyse & visualisations
Analyse du sentiment par **président**, par **année** et par **bloc politique**.

In [ ]:
# ── Mapping bloc politique ────────────────────────────────────────────────────
GAUCHE_KW = [
    "socialiste", "communiste", "lutte ouvrière", "parti socialiste unifié",
    "radicaux de gauche", "sfio", " ps;", ";ps;", "\nps\n", "gauche",
    "trotsk", "ligue communiste", "nouvelle gauche",
]
DROITE_KW = [
    "rassemblement pour la république", "rpr", "républicain indépendant",
    "union des républicains", "front national", "alliance républicaine",
    "gaulliste", "cnip", "droite",
]
CENTRE_KW = [
    "union pour la démocratie", "udf", "centre démocrate",
    "mouvement républicain populaire", "mrp", "cds", "centre",
    "réformateur", "libéral", "radical",
]

def get_bloc(parti: str) -> str:
    if not isinstance(parti, str):
        return "non classé"
    p = parti.lower()
    # Priorité gauche > droite > centre pour éviter les faux positifs
    if any(kw in p for kw in GAUCHE_KW):
        return "gauche"
    if any(kw in p for kw in DROITE_KW):
        return "droite"
    if any(kw in p for kw in CENTRE_KW):
        return "centre"
    return "non classé"

df_annotated = pd.read_excel(ANNOTATED_FILE)
df_final = df_annotated[
    df_annotated["sentiment_president"].isin(["positif", "négatif", "neutre"])
].copy()
df_final["bloc_politique"] = df_final["parti"].apply(get_bloc)

print(f"Dataset final : {len(df_final):,} lignes")
print("\nBloc politique :")
print(df_final["bloc_politique"].value_counts().to_string())


In [ ]:
# ── Graphique 1 : Sentiment par président ─────────────────────────────────────
pivot_pres = (
    df_final.groupby(["actual_president", "sentiment_president"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=LABEL_LIST, fill_value=0)
)
# Normaliser en pourcentages
pivot_pres_pct = pivot_pres.div(pivot_pres.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comptes bruts
pivot_pres.plot(kind="bar", ax=axes[0], color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
axes[0].set_title("Mentions par président (comptes bruts)")
axes[0].set_xlabel("")
axes[0].set_ylabel("Nombre de mentions")
axes[0].legend(title="Sentiment")
axes[0].tick_params(axis="x", rotation=20)

# Pourcentages
pivot_pres_pct.plot(kind="bar", ax=axes[1], color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
axes[1].set_title("Mentions par président (pourcentages)")
axes[1].set_xlabel("")
axes[1].set_ylabel("% de mentions")
axes[1].legend(title="Sentiment")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "sentiment_par_president.png", dpi=150)
plt.show()
print(f"Graphique sauvegardé : {GRAPHS_DIR / 'sentiment_par_president.png'}")


In [ ]:
# ── Graphique 2 : Sentiment par année ────────────────────────────────────────
pivot_annee = (
    df_final.groupby(["annee", "sentiment_president"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=LABEL_LIST, fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
pivot_annee.plot(kind="bar", stacked=True, ax=ax, color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
ax.set_title("Sentiment envers le président par année électorale")
ax.set_xlabel("Année")
ax.set_ylabel("Nombre de mentions")
ax.legend(title="Sentiment")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "sentiment_par_annee.png", dpi=150)
plt.show()
print(f"Graphique sauvegardé : {GRAPHS_DIR / 'sentiment_par_annee.png'}")


In [ ]:
# ── Graphique 3 : Sentiment par bloc politique ────────────────────────────────
blocs_order = ["gauche", "centre", "droite", "non classé"]
pivot_bloc = (
    df_final.groupby(["bloc_politique", "sentiment_president"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=blocs_order, columns=LABEL_LIST, fill_value=0)
)
pivot_bloc_pct = pivot_bloc.div(pivot_bloc.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_bloc.plot(kind="bar", ax=axes[0], color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
axes[0].set_title("Sentiment par bloc politique (comptes bruts)")
axes[0].set_xlabel("")
axes[0].set_ylabel("Nombre de mentions")
axes[0].legend(title="Sentiment")
axes[0].tick_params(axis="x", rotation=10)

pivot_bloc_pct.plot(kind="bar", ax=axes[1], color=[COLORS[l] for l in LABEL_LIST], edgecolor="white")
axes[1].set_title("Sentiment par bloc politique (pourcentages)")
axes[1].set_xlabel("")
axes[1].set_ylabel("% de mentions")
axes[1].legend(title="Sentiment")
axes[1].tick_params(axis="x", rotation=10)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "sentiment_par_bloc.png", dpi=150)
plt.show()
print(f"Graphique sauvegardé : {GRAPHS_DIR / 'sentiment_par_bloc.png'}")


In [ ]:
# ── Graphique 4 : Heatmap sentiment × président × bloc ───────────────────────
pivot_heat = (
    df_final[df_final["bloc_politique"] != "non classé"]
    .groupby(["actual_president", "bloc_politique", "sentiment_president"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=LABEL_LIST, fill_value=0)
)

# Taux de sentiment positif (%)
pivot_heat["pct_positif"] = (
    pivot_heat["positif"] / pivot_heat.sum(axis=1) * 100
).round(1)

print("Taux de sentiment positif (%) par président × bloc politique :")
print(pivot_heat["pct_positif"].unstack().to_string())

fig, ax = plt.subplots(figsize=(8, 4))
heatmap_data = pivot_heat["pct_positif"].unstack().fillna(0)
sns.heatmap(
    heatmap_data, annot=True, fmt=".1f", cmap="RdYlGn",
    vmin=0, vmax=100, ax=ax, linewidths=0.5,
)
ax.set_title("% mentions positives envers le président\n(par président × bloc politique)")
ax.set_xlabel("Bloc politique")
ax.set_ylabel("Président")
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "heatmap_president_bloc.png", dpi=150)
plt.show()
print(f"Graphique sauvegardé : {GRAPHS_DIR / 'heatmap_president_bloc.png'}")


In [ ]:
# ── Sauvegarde du tableau final ───────────────────────────────────────────────
df_final.to_excel(FINAL_FILE, index=False, engine="openpyxl")
print(f"Fichier final sauvegardé : {FINAL_FILE}")
print(f"  Lignes     : {len(df_final):,}")
print(f"  Colonnes   : {df_final.columns.tolist()}")
print()
print("Distribution finale des sentiments :")
print(df_final["sentiment_president"].value_counts().to_string())
print()
print("Graphiques enregistrés dans :", GRAPHS_DIR)
for f in sorted(GRAPHS_DIR.glob("*.png")):
    print(f"  {f.name}")
